<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c37-header.png" alt="Nextia Learning · Recommendation and Personalisation: Show Each Person What Matters" width="100%">

# Monitor and retrain

**[Monitor and retrain](https://learning.nextia-ai.com/courses/recommendations/m09/monitor-and-retrain/)** · Recommendation and Personalisation: Show Each Person What Matters · Module 9, lesson 2 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** watch a recommender week by week, by segment, see how a model that is never retrained goes stale, and compare a new model with the current one before release.

| | |
|---|---|
| **Time** | About 30 minutes |
| **Needs** | An internet connection for the setup cells (about 7 MB). No account and no key. |
| **You should know** | Batch serving and the fallback, from [Batch or real time](https://learning.nextia-ai.com/courses/recommendations/m09/batch-or-real-time/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1, SciPy 1.18.1 and matplotlib 3.11.2 (implicit 0.7.3 or LightGBM 4.7.0 where the lesson needs it), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/recommendations/m09/monitor-and-retrain/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C37/M09-L02-monitor-and-retrain/monitor-and-retrain-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, scipy 1.18.1, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 scipy==1.18.1 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'pandas', 'scipy']:
    print(name, version(name))

## Setup: get the Larkfield data

Run the next cell. It downloads the files that this lesson needs (about 7 MB) into a folder `larkfield-recs` and checks each file's checksum. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-recs` (in Colab: the **Files** panel on the left), then run the cell again.

Where the data comes from: Larkfield is an invented shop, and its shop log is **synthetic**: our code made the products, the customers and every view, basket and order, with a model of customer behaviour. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C37/data/dataset-card.md) explains how.

In [ ]:
# Setup: download the files into larkfield-recs/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C37/"
FILES = {
    "data/products.csv": "3da37a3809f6067da12eb5778b8abcf809545599db6959208c72a790861407ff",
    "data/events.csv.gz": "3f46ddfff683586ab1548bc0e6756d85a439a1de941e6de3030edc383d06d1b3",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-recs").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-recs and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in larkfield-recs.")

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd`, SciPy's sparse matrices as `sp` and matplotlib. It defines the course's small helpers, the same ideas as the course project:

- `products()`, `customers()` and `events()` read the data. `events()` has one row per view, basket add or order, with the column `day` (days since 1 February 2025).
- `split(period)` gives the events before the period (`train`, to learn from) and, for each customer who ordered in the period, the products they ordered (`truth`). The periods are `"validation"`, `"test"` and `"launch"`.
- `recall_at_k(list, ordered, k)`, `ndcg_at_k(list, ordered, k)` and `coverage(lists, n_products)` measure suggestion lists.
- `show(ids)` prints products with their title and subcategory.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the data, the time split, the measures, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
START = pd.Timestamp("2025-02-01")

def products():
    """One row per product (2,994)."""
    df = pd.read_csv("data/products.csv", keep_default_na=False)
    for col in ("age_restricted", "sensitive", "in_stock"):
        df[col] = df[col].astype(str).eq("True")
    return df

def customers():
    """One row per customer (40,000)."""
    return pd.read_csv("data/customers.csv")

def events():
    """One row per event: view, basket or order. day = days since 1 February 2025."""
    ev = pd.read_csv("data/events.csv.gz", parse_dates=["timestamp"])
    ev["day"] = (ev["timestamp"] - START).dt.days
    return ev

def split(period, ev=None):
    """train = the events before the period; truth = {customer: set of products ordered in the period}."""
    ev = events() if ev is None else ev
    first, last = json.load(open("data/splits.json"))[period]
    a, b = (pd.Timestamp(first) - START).days, (pd.Timestamp(last) - START).days
    train = ev[ev["day"] < a]
    target = ev[(ev["day"] >= a) & (ev["day"] <= b) & (ev["event"] == "order")]
    truth = target.groupby("customer_id")["product_id"].agg(set).to_dict()
    return train, truth

def recall_at_k(rec, ordered, k=10):
    """Share of the ordered products that are in the first k suggestions."""
    return len(set(list(rec)[:k]) & set(ordered)) / len(ordered)

def ndcg_at_k(rec, ordered, k=10):
    """DCG of the list divided by the best possible DCG (discount log2(place + 1))."""
    gains = [1.0 if p in ordered else 0.0 for p in list(rec)[:k]]
    dcg = sum(g / np.log2(i + 2) for i, g in enumerate(gains))
    ideal = sum(1 / np.log2(i + 2) for i in range(min(len(ordered), k)))
    return dcg / ideal

def coverage(lists, n_products, k=10):
    """Share of the catalogue that appears in at least one list."""
    return len({p for rec in lists for p in list(rec)[:k]}) / n_products

def rounded(x, n=3):
    """x rounded to n places (None or ... unchanged: the task is not done yet)."""
    return x if x is None or x is ... else round(float(x), n)

def show(ids, n=10):
    """Print products: place, ID, title, subcategory."""
    global _PR
    if "_PR" not in globals():
        _PR = products().set_index("product_id")
    for i, p in enumerate(list(ids)[:n], 1):
        print(f"{i:2}. {p}  {_PR.loc[p, 'title'][:52]:<52}  {_PR.loc[p, 'subcategory']}")

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Eight weeks of orders

A recommender is trained once, then it serves customers for days or weeks. Meanwhile the shop changes: a new summer range, the end of a season, new customers. **Monitoring** means measuring the running system again and again, and comparing the numbers with what you expect.

The next cell prepares the last eight weeks of the log. For each week, it finds the customers who ordered, what they ordered, and their segment: **new** (no order before the week), **occasional** (1 or 2) or **loyal** (3 or more). A list of the trending products is the model: it is the fallback of the service, and it is cheap to recompute.

In [ ]:
ev = events()
pr = products()
added = dict(zip(pr["product_id"], (pd.to_datetime(pr["added"]) - START).dt.days))
last = ev["day"].max()
orders = ev[ev["event"] == "order"]

def trending(first_day, n=10):
    """The products that most customers ordered in the 28 days before first_day, among those on sale."""
    o = orders[(orders["day"] < first_day) & (orders["day"] >= first_day - 28)]
    count = o.drop_duplicates(["customer_id", "product_id"]).groupby("product_id").size()
    count = count.sort_values(ascending=False, kind="stable")
    return [p for p in count.index if added[p] <= first_day][:n]

def week(first_day):
    """The customers who ordered in the 7 days from first_day, what they ordered, and their segment."""
    o = orders[(orders["day"] >= first_day) & (orders["day"] <= min(first_day + 6, last))]
    truth = o.groupby("customer_id")["product_id"].agg(set)
    before = orders[orders["day"] < first_day].groupby("customer_id")["session_id"].nunique()
    n = before.reindex(truth.index).fillna(0)
    segment = np.where(n == 0, "new", np.where(n <= 2, "occasional", "loyal"))
    return pd.DataFrame({"truth": truth, "segment": segment})

starts = [last - 7 * 8 + 1 + 7 * w for w in range(8)]            # the last 8 weeks of the log
print("weeks from", [str((START + pd.Timedelta(days=int(s))).date()) for s in starts])

## 2. A fresh list and a frozen list

> **Predict.** A trending list computed on 11 May and never computed again: how good is it seven weeks later, at the end of June?

The next cell measures two lists every week. The **fresh** list is computed again at the start of each week (retrained). The **frozen** list is the one of the first week, never computed again (a stale model). It prints recall@10 by segment for the fresh list, and for all customers for both lists.

In [ ]:
frozen = trending(starts[0])                  # computed once, in the first week, never again
rows = []
for s in starts:
    w = week(s)
    fresh = trending(s)                        # retrained: computed again every week
    w["fresh"] = [recall_at_k(fresh, t) for t in w["truth"]]
    w["frozen"] = [recall_at_k(frozen, t) for t in w["truth"]]
    by = w.groupby("segment")["fresh"].mean()
    rows.append({"week": str((START + pd.Timedelta(days=int(s))).date()), "customers": len(w),
                 "new": by.get("new"), "occasional": by.get("occasional"), "loyal": by.get("loyal"),
                 "fresh (all)": w["fresh"].mean(), "frozen (all)": w["frozen"].mean()})
monitor = pd.DataFrame(rows).set_index("week")
monitor.round(3)

In [ ]:
ax = monitor[["new", "occasional", "loyal"]].plot(marker="o", figsize=(8, 3.5))
monitor["frozen (all)"].plot(ax=ax, color="grey", linestyle="--", label="frozen list (all)")
ax.set_ylabel("recall@10 in the week"); ax.set_ylim(0, 0.3); ax.legend(); ax.set_title("Trending, recomputed every week, by segment")
plt.show()

Read the segment lines first. They jump from week to week: each segment has only a few hundred customers in a week. One bad week is not a reason to act. The dashed line is the frozen list. In the first week, it is the same as the fresh list. Then it falls behind.

## 3. Your turn: what does the stale list cost?

Compute how much recall@10 the frozen list loses against the fresh list in the last week, for all customers.

In [ ]:
last_week = monitor.iloc[-1]
loss = ...      # recall of the fresh list minus recall of the frozen list, in the last week (all customers)
print(rounded(loss))

In [ ]:
expect('loss', rounded(loss), 0.055, tolerance=0.001)

## 4. Compare before you release

A retrained model is not automatically better. Before it replaces the current one, compare both on the same customers and the same week, segment by segment. The simple rule below releases the new model only if no segment gets worse.

In [ ]:
def release(new, current, tolerance=0.0):
    """Release the new model only if no segment is worse than with the current one."""
    worse = [s for s in ("new", "occasional", "loyal") if new[s] < current[s] - tolerance]
    return ("release" if not worse else "keep the current model"), worse

w = week(starts[-1])
fresh, old = trending(starts[-1]), frozen
new_by = {s: np.mean([recall_at_k(fresh, t) for t in w["truth"][w["segment"] == s]]) for s in ("new", "occasional", "loyal")}
old_by = {s: np.mean([recall_at_k(old, t) for t in w["truth"][w["segment"] == s]]) for s in ("new", "occasional", "loyal")}
print("new model:", {k: round(v, 3) for k, v in new_by.items()})
print("current:  ", {k: round(v, 3) for k, v in old_by.items()})
print(release(new_by, old_by))

> **Check.** In the last week, the fresh list has recall@10 0.152 and the frozen list 0.098 (all customers). The rule releases the fresh list. In the project, `recs monitor --method trending --weeks 8` prints almost the same table (one week differs a little, because two products with the same count can come in another order). The same comparison with ALS is `recs monitor --weeks 8`.

A real release check also looks at coverage, short or empty lists, and speed. The lesson shows them for Larkfield's ALS model.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Privacy and sensitive suggestions](https://learning.nextia-ai.com/courses/recommendations/m09/privacy-and-sensitive-suggestions/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/recommendations/m09/monitor-and-retrain/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The Larkfield shop log is synthetic: our code made the products, the customers and everything they did, with a model of customer behaviour. No real customer is in this data.